# 실습 18 · 단백질 설계 심화 — ESM-2 로 변이 스캐닝 & 단백질 공학
### 항체·효소를 '어디를 바꾸면 좋은지' AI 에게 묻기

**왜 중요한가 (바이오의약품·항체 시대의 핵심)**
바이오시밀러·항체·효소 의약품에서 **단백질 공학**(원하는 성질을 갖도록
아미노산을 바꾸는 것)은 핵심 기술입니다. 문제는 후보 변이의 수가
**천문학적**(길이 100이면 100×19=1900가지, 조합은 폭발적)이라는 점.
ESM-2 같은 **단백질 언어모델**은 수억 개 자연 단백질을 학습했기에,
**"어떤 아미노산이 그 자리에 자연스러운가"** 를 알고 있어, 실험 전에
유망한 변이를 **컴퓨터로 먼저 걸러줍니다.**

**이 노트북에서 배우는 것 (예제 15의 ESM 파트 심화)**
1. **단일 forward pass** 로 서열 전체의 **변이 점수 지도(deep mutational scan)** 만들기
2. **히트맵**으로 어디가 보존적(못 바꿈)이고 어디가 유연(바꿔도 됨)인지 시각화
3. 가장 **해로운/허용되는** 변이 Top 목록 뽑기 → 실험 우선순위
4. ESM **임베딩**으로 단백질 간 유사도·기능 관계 파악
5. 항체 최적화·효소 안정화에서의 활용 패턴

> 💡 런타임을 **T4 GPU** 로 바꾸면 빠릅니다. (CPU 로도 동작, 소형 모델 사용)


In [ ]:
!pip install transformers torch matplotlib -q
import torch, numpy as np, matplotlib.pyplot as plt
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForMaskedLM
device = "cuda" if torch.cuda.is_available() else "cpu"
print("준비 완료 ✅  (device:", device, ")")


## 1. ESM-2 단백질 언어모델 로드 (마스크 예측용)
예제 15와 같은 경량 ESM-2 를 씁니다. 이번엔 **각 위치에서 20종 아미노산의
확률**을 뽑는 `MaskedLM` 헤드를 사용합니다. 이 확률이 변이 점수의 근거입니다.


In [ ]:
esm_name = "facebook/esm2_t12_35M_UR50D"
tok = AutoTokenizer.from_pretrained(esm_name)
mlm = AutoModelForMaskedLM.from_pretrained(esm_name).to(device).eval()

# 예제 단백질 서열 (데모용 70여 잔기). 실무에선 항체 CDR·효소 활성부위를 넣음
seq = ("MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKVKALPDAQFEVVHSLAKWKR")
print("서열 길이:", len(seq), "잔기")
print("ESM-2 로드 완료 ✅")


## 2. ⭐ 변이 점수 지도 — 단일 forward pass 로 전체 스캔
표준 기법(**wild-type marginals**): 야생형 서열을 **한 번만** 모델에 넣고,
각 위치에서 20종 아미노산의 로그확률을 얻습니다.
변이 점수 = log P(변이 아미노산) − log P(야생형).
**점수가 음수로 클수록 '해로운(비자연스러운)' 변이**로 예측됩니다.


In [ ]:
AAs = list("ACDEFGHIKLMNPQRSTVWY")   # 표준 20 아미노산

# 야생형 서열 1회 통과 → 모든 위치의 로그확률 확보
enc = tok(seq, return_tensors="pt").to(device)
with torch.no_grad():
    logits = mlm(**enc).logits[0]                 # (토큰수, 어휘)
logp = F.log_softmax(logits, dim=-1)              # 로그확률

aa_ids = {a: tok.convert_tokens_to_ids(a) for a in AAs}
L = len(seq)
score = np.zeros((20, L))                          # 20 x 길이
for j, wt in enumerate(seq):
    pos_tok = j + 1                                # [CLS] 보정
    wt_lp = logp[pos_tok, aa_ids[wt]].item()
    for i, a in enumerate(AAs):
        score[i, j] = logp[pos_tok, aa_ids[a]].item() - wt_lp
print("변이 점수 지도 shape:", score.shape, "(20 아미노산 x", L, "위치)")
print("→ 단 한 번의 모델 실행으로 모든 변이를 점수화했습니다")


## 3. ⭐ 히트맵 — 어디를 바꿔도 되고, 어디는 절대 안 되는지
파란색(음수)=해로운 변이, 빨간색(0 근처)=허용 가능.
**세로 줄 전체가 파란 위치 = 보존적(기능에 필수)**,
얼룩덜룩한 위치 = 변이 허용도가 높은 곳입니다.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(score, aspect="auto", cmap="RdBu", vmin=-8, vmax=2)
ax.set_yticks(range(20)); ax.set_yticklabels(AAs)
ax.set_xticks(range(0, L, 5)); ax.set_xticklabels(range(1, L+1, 5))
ax.set_xlabel("서열 위치"); ax.set_ylabel("변이 아미노산")
ax.set_title("ESM-2 변이 효과 지도 (파랑=해로움, 빨강=허용)")
plt.colorbar(im, label="변이 점수 (log-likelihood ratio)")
plt.tight_layout(); plt.show()


## 4. 보존 위치 vs 유연 위치 찾기 (단백질 공학 힌트)
위치별 **평균 변이 점수**가 낮을수록(=대부분 변이가 해로움) 보존적이고 중요한
자리입니다. 반대로 높은 위치는 자유롭게 엔지니어링할 수 있는 후보입니다.


In [ ]:
pos_mean = score.mean(axis=0)                      # 위치별 평균 변이 점수
order = np.argsort(pos_mean)

print("■ 가장 보존적인(건드리면 안 되는) 위치 Top5")
for j in order[:5]:
    print(f"   위치 {j+1:2d} ({seq[j]}): 평균점수 {pos_mean[j]:+.2f}  → 기능에 필수 가능성")

print("\n■ 가장 유연한(엔지니어링 여지 큰) 위치 Top5")
for j in order[::-1][:5]:
    print(f"   위치 {j+1:2d} ({seq[j]}): 평균점수 {pos_mean[j]:+.2f}  → 변이 허용도 높음")


## 5. 유망한 변이 / 위험한 변이 Top 목록
개별 변이 단위로도 순위를 매길 수 있습니다. 실험을 어디부터 할지 정하는
**우선순위 리스트**가 됩니다 (항체 친화도 개선 후보 등).


In [ ]:
records = []
for j, wt in enumerate(seq):
    for i, a in enumerate(AAs):
        if a != wt:
            records.append((f"{wt}{j+1}{a}", score[i, j]))
records.sort(key=lambda x: x[1])

print("■ 가장 해로울 것으로 예측되는 변이 Top5 (피해야 할 것)")
for name, s in records[:5]:
    print(f"   {name}: {s:+.2f}")
print("\n■ 가장 허용/선호될 변이 Top5 (개선 후보로 실험 우선)")
for name, s in records[::-1][:5]:
    print(f"   {name}: {s:+.2f}")


## 6. ESM 임베딩으로 단백질 관계 보기
임베딩(의미 벡터)은 서열이 달라도 **기능이 비슷하면 가까이** 위치합니다.
서로 다른 단백질들의 유사도를 비교해 봅니다 (기능 예측·분류의 기반).


In [ ]:
from transformers import AutoModel
emb_model = AutoModel.from_pretrained(esm_name).to(device).eval()

def embed(s):
    e = tok(s, return_tensors="pt").to(device)
    with torch.no_grad():
        h = emb_model(**e).last_hidden_state[0]
    return h.mean(0).cpu().numpy()                 # 평균 풀링 = 단백질 1개 벡터

seqs = {
 "원본":       seq,
 "보존변이(유해)": seq[:4] + "P" + seq[5:],         # 보존위치를 Pro로 (해로울 것)
 "유연변이(무해)": seq[:0] + seq,                   # 거의 동일
 "무관단백질":   "MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKT",  # 인슐린 일부
}
names = list(seqs); V = np.stack([embed(s) for s in seqs.values()])
Vn = V / np.linalg.norm(V, axis=1, keepdims=True)
sim = Vn @ Vn.T
print("코사인 유사도 행렬:")
print("        " + "  ".join(f"{n[:6]:>6}" for n in names))
for i, n in enumerate(names):
    print(f"{n[:6]:>6} " + "  ".join(f"{sim[i,j]:6.2f}" for j in range(len(names))))
print("\n→ 원본과 무관단백질은 유사도가 낮고, 유사 서열끼리는 높게 나옵니다")


## 7. 단백질 언어모델 활용 지형도 (참고)
| 목적 | 방법 | 현장 예 |
|---|---|---|
| **변이 효과 예측** | ESM 마스크 확률 (이 예제) | 항체 친화도·안정성 개선 후보 선별 |
| **기능·분류 예측** | ESM 임베딩 + 분류기 | 효소 기능, 세포 위치 예측 |
| **구조 예측** | ESMFold (예제 13) | 서열→3D, 결합부위 파악 |
| **서열 생성/설계** | ProtGPT2, ESM-IF | 신규 단백질·효소 de novo 설계 |

**표준 워크플로우**: 변이 스캔으로 후보 축소 → 임베딩·구조로 검증 →
소수 후보만 실제 실험 → **실험 횟수·비용 대폭 절감**


## 정리 & 현장 응용
- **ESM-2 변이 스캐닝**: 단 한 번의 모델 실행으로 서열 전체 변이 지도를 얻는다
- **보존/유연 위치 구분**: 어디는 지키고 어디를 엔지니어링할지 실험 전에 판단
- **임베딩**: 서열→기능 관계 파악 → 기능 예측·분류의 입력
- 항체 최적화, 효소 안정화, 바이오시밀러 개발에서 **실험 후보를 대폭 축소**
- 예제 13(구조예측)·15(임베딩)와 연결하면 **단백질 AI 파이프라인** 완성
- **면접 한 문장**: "ESM-2 로 단백질 변이 효과를 한 번에 스캔해 보존 위치와
  개선 후보를 골라내면, 항체·효소 공학의 실험 후보를 크게 줄일 수 있습니다."
